# Hotel Review Classification — Training (Kaggle)

Trains two separate `bert-base-uncased` models on the HRAST hotel-review
dataset:

1. A **sentiment model** that predicts one label per sentence: `negative`,
   `neutral`, or `positive`.
2. An **aspect model** that predicts which of 21 hotel aspects (Staff,
   Breakfast, Wi-Fi, ...) a sentence mentions. A sentence can mention more
   than one aspect.

The code is written plainly: normal `for` loops, small functions, one job
per cell, no advanced tricks.

## Before you run this

1. **Turn on a GPU.** Notebook settings (right panel) → Accelerator → GPU.
2. **Turn on Internet.** Needed to download `bert-base-uncased` and install
   one small package.
3. **Attach the dataset.** Add Data → upload `dataset/HRAST.csv` from this
   project as a new Kaggle Dataset, then attach it here. The notebook finds
   it automatically under `/kaggle/input/`.

## After it finishes

Kaggle keeps everything this notebook writes under `/kaggle/working` as its
**Output** once you save/commit a version. To run inference later without
retraining, open **kaggle_infer.ipynb**, attach this notebook's Output as an
input dataset, and point it at the saved `artifacts/sentiment` and
`artifacts/aspects` folders.


## Step 0: Install packages and check the GPU

In [ ]:
# Kaggle already has PyTorch and Transformers installed.
# We only need this one extra package, for a stratified train/val/test split.
!pip install --quiet iterative-stratification==0.1.9


In [ ]:
import re
import json
import glob
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


In [ ]:
# Use the GPU if Kaggle gave us one, otherwise fall back to CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("No GPU found. Training will be slow on CPU.")


In [ ]:
# Fixed seed so the split and training are reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
# Kaggle keeps everything written under /kaggle/working as this notebook's output.
WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data"
ARTIFACTS_DIR = WORKING_DIR / "artifacts"
SENTIMENT_DIR = ARTIFACTS_DIR / "sentiment"
ASPECT_DIR = ARTIFACTS_DIR / "aspects"

DATA_DIR.mkdir(parents=True, exist_ok=True)
SENTIMENT_DIR.mkdir(parents=True, exist_ok=True)
ASPECT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Find the HRAST.csv file from the dataset you attached under /kaggle/input.
csv_files = glob.glob("/kaggle/input/**/HRAST.csv", recursive=True)
if len(csv_files) == 0:
    raise FileNotFoundError("HRAST.csv not found under /kaggle/input. Did you attach the dataset?")
RAW_CSV_PATH = csv_files[0]
print("Found dataset at:", RAW_CSV_PATH)


## Step 1: Define the labels

Sentiment is one label per sentence. Aspects are 21 yes/no labels per sentence.

In [ ]:
SENTIMENT_COLUMNS = ["positive", "negative", "neutral"]
SENTIMENT_LABELS = ["negative", "neutral", "positive"]
SENTIMENT_TO_ID = {"negative": 0, "neutral": 1, "positive": 2}

ASPECT_COLUMNS = [
    "Clean", "Comfort", "Facilities/Amenities", "Location",
    "Restaurant (dinner)", "Staff", "View (Balcony)", "Breakfast", "Room",
    "Pool", "Beach", "Bathroom/Shower (toilet)", "Bar", "Bed", "Parking",
    "Noise", "Reception-checkin", "Lift", "Value for money", "Wi-Fi", "Generic",
]

LABEL_COLUMNS = SENTIMENT_COLUMNS + ASPECT_COLUMNS


## Step 2: Load and clean the data

We go through the raw file one simple check at a time: drop the empty
column, tidy up the text, remove rows with bad labels, and remove duplicate
reviews that disagree on their labels.


In [ ]:
df = pd.read_csv(RAW_CSV_PATH, encoding="utf-8-sig")
print("Rows in raw file:", len(df))

# The raw file has one empty, unnamed column. Drop it.
for column in df.columns:
    if str(column).startswith("Unnamed"):
        df = df.drop(columns=[column])


In [ ]:
def clean_text(text):
    # Collapse repeated spaces/tabs/newlines into a single space.
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["review"] = df["review"].apply(clean_text)
df = df[df["review"] != ""]
print("Rows after removing empty reviews:", len(df))


In [ ]:
# Every sentiment/aspect cell must be exactly 0 or 1. Drop rows where it is not.
def row_labels_are_binary(row):
    for column in LABEL_COLUMNS:
        if row[column] not in (0, 1, "0", "1"):
            return False
    return True

keep_mask = df.apply(row_labels_are_binary, axis=1)
df = df[keep_mask]
print("Rows after removing non-binary label values:", len(df))


In [ ]:
# Every row must have exactly one sentiment label.
for column in SENTIMENT_COLUMNS:
    df[column] = df[column].astype(int)
for column in ASPECT_COLUMNS:
    df[column] = df[column].astype(int)

sentiment_total = df["positive"] + df["negative"] + df["neutral"]
df = df[sentiment_total == 1]
print("Rows after keeping exactly one sentiment label:", len(df))


In [ ]:
# Turn the three sentiment columns into one readable label.
def sentiment_name(row):
    if row["negative"] == 1:
        return "negative"
    if row["neutral"] == 1:
        return "neutral"
    return "positive"

df["sentiment"] = df.apply(sentiment_name, axis=1)


In [ ]:
# If the same review text shows up more than once, only keep it when every
# copy has the exact same labels. If copies disagree, we cannot trust which
# one is right, so we drop all of them.
rows_to_keep = []

for review_text, group in df.groupby("review"):
    first_row_labels = group[LABEL_COLUMNS].iloc[0].tolist()
    all_copies_agree = True
    for row_number in range(len(group)):
        this_row_labels = group[LABEL_COLUMNS].iloc[row_number].tolist()
        if this_row_labels != first_row_labels:
            all_copies_agree = False
    if all_copies_agree:
        rows_to_keep.append(group.iloc[0])

df = pd.DataFrame(rows_to_keep).reset_index(drop=True)
print("Rows after removing duplicate reviews:", len(df))


In [ ]:
print("Final cleaned rows:", len(df))
# This exact number is documented in the project plan for this dataset.
assert len(df) == 23095, f"Expected 23095 clean rows, got {len(df)}"


## Step 3: Split into train / validation / test

We use a 70/15/15 split. `MultilabelStratifiedShuffleSplit` keeps rare
aspects and the small neutral-sentiment class represented in every split.


In [ ]:
stratify_columns = ASPECT_COLUMNS + SENTIMENT_COLUMNS
label_matrix = df[stratify_columns].to_numpy()

# First split off 70% for training, 30% left over.
splitter_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_index, rest_index = next(splitter_1.split(df, label_matrix))

train_df = df.iloc[train_index].reset_index(drop=True)
rest_df = df.iloc[rest_index].reset_index(drop=True)

# Split the remaining 30% evenly into validation (15%) and test (15%).
rest_label_matrix = rest_df[stratify_columns].to_numpy()
splitter_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
validation_index, test_index = next(splitter_2.split(rest_df, rest_label_matrix))

validation_df = rest_df.iloc[validation_index].reset_index(drop=True)
test_df = rest_df.iloc[test_index].reset_index(drop=True)

print("Train rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))


In [ ]:
train_df.to_csv(DATA_DIR / "train.csv", index=False)
validation_df.to_csv(DATA_DIR / "validation.csv", index=False)
test_df.to_csv(DATA_DIR / "test.csv", index=False)


## Step 4: Training settings

Same hyperparameters for both models.

In [ ]:
BASE_MODEL = "bert-base-uncased"
MAX_LENGTH = 128
EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32


### A small helper used by both models

Splits a list of texts and labels into fixed-size batches, in random order for training and in order for evaluation.

In [ ]:
def make_batches(texts, labels, batch_size, shuffle):
    indices = list(range(len(texts)))
    if shuffle:
        random.shuffle(indices)

    batches = []
    start = 0
    while start < len(indices):
        end = start + batch_size
        batch_indices = indices[start:end]
        batch_texts = [texts[i] for i in batch_indices]
        batch_labels = [labels[i] for i in batch_indices]
        batches.append((batch_texts, batch_labels))
        start = end

    return batches


## Step 5: Sentiment model — prepare the data and the model

In [ ]:
sentiment_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_texts = train_df["review"].tolist()
validation_texts = validation_df["review"].tolist()
test_texts = test_df["review"].tolist()

train_sentiment_labels = []
for name in train_df["sentiment"]:
    train_sentiment_labels.append(SENTIMENT_TO_ID[name])

validation_sentiment_labels = []
for name in validation_df["sentiment"]:
    validation_sentiment_labels.append(SENTIMENT_TO_ID[name])

test_sentiment_labels = []
for name in test_df["sentiment"]:
    test_sentiment_labels.append(SENTIMENT_TO_ID[name])


In [ ]:
sentiment_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=3)
sentiment_model.to(device)

sentiment_optimizer = torch.optim.AdamW(sentiment_model.parameters(), lr=LEARNING_RATE)
sentiment_loss_function = torch.nn.CrossEntropyLoss()


### Training and evaluation functions for the sentiment model

In [ ]:
def train_sentiment_one_epoch(texts, labels):
    sentiment_model.train()
    batches = make_batches(texts, labels, TRAIN_BATCH_SIZE, shuffle=True)
    total_loss = 0.0

    for batch_texts, batch_labels in batches:
        encoded = sentiment_tokenizer(
            batch_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        labels_tensor = torch.tensor(batch_labels).to(device)

        sentiment_optimizer.zero_grad()
        output = sentiment_model(**encoded)
        loss = sentiment_loss_function(output.logits, labels_tensor)
        loss.backward()
        sentiment_optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(batches)
    return average_loss


In [ ]:
def evaluate_sentiment(texts, labels):
    sentiment_model.eval()
    batches = make_batches(texts, labels, EVAL_BATCH_SIZE, shuffle=False)

    all_predictions = []
    all_true_labels = []

    with torch.no_grad():
        for batch_texts, batch_labels in batches:
            encoded = sentiment_tokenizer(
                batch_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
            )
            encoded = {key: value.to(device) for key, value in encoded.items()}
            output = sentiment_model(**encoded)
            predictions = torch.argmax(output.logits, dim=1).cpu().tolist()

            all_predictions.extend(predictions)
            all_true_labels.extend(batch_labels)

    accuracy = accuracy_score(all_true_labels, all_predictions)
    macro_f1 = f1_score(all_true_labels, all_predictions, average="macro")
    return accuracy, macro_f1


### Train the sentiment model

After every epoch we check validation macro-F1 and keep only the best checkpoint.

In [ ]:
best_sentiment_macro_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_sentiment_one_epoch(train_texts, train_sentiment_labels)
    val_accuracy, val_macro_f1 = evaluate_sentiment(validation_texts, validation_sentiment_labels)

    print(f"Epoch {epoch}: train loss = {train_loss:.4f}, "
          f"validation accuracy = {val_accuracy:.4f}, validation macro-F1 = {val_macro_f1:.4f}")

    if val_macro_f1 > best_sentiment_macro_f1:
        best_sentiment_macro_f1 = val_macro_f1
        sentiment_model.save_pretrained(SENTIMENT_DIR)
        sentiment_tokenizer.save_pretrained(SENTIMENT_DIR)
        print("  New best sentiment model saved.")


In [ ]:
# Reload the best checkpoint (not necessarily the last epoch) and score it once on the test set.
sentiment_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_DIR)
sentiment_model.to(device)

test_accuracy, test_macro_f1 = evaluate_sentiment(test_texts, test_sentiment_labels)
print(f"Sentiment test accuracy = {test_accuracy:.4f}, test macro-F1 = {test_macro_f1:.4f}")


## Step 6: Aspect model — prepare the data and the model

In [ ]:
aspect_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

train_aspect_labels = train_df[ASPECT_COLUMNS].values.tolist()
validation_aspect_labels = validation_df[ASPECT_COLUMNS].values.tolist()
test_aspect_labels = test_df[ASPECT_COLUMNS].values.tolist()


In [ ]:
aspect_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(ASPECT_COLUMNS), problem_type="multi_label_classification"
)
aspect_model.to(device)

aspect_optimizer = torch.optim.AdamW(aspect_model.parameters(), lr=LEARNING_RATE)
aspect_loss_function = torch.nn.BCEWithLogitsLoss()


### Training and evaluation functions for the aspect model

Same idea as the sentiment model, but the loss is `BCEWithLogitsLoss` (one yes/no decision per aspect) instead of cross-entropy.

In [ ]:
def train_aspect_one_epoch(texts, labels):
    aspect_model.train()
    batches = make_batches(texts, labels, TRAIN_BATCH_SIZE, shuffle=True)
    total_loss = 0.0

    for batch_texts, batch_labels in batches:
        encoded = aspect_tokenizer(
            batch_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        labels_tensor = torch.tensor(batch_labels, dtype=torch.float32).to(device)

        aspect_optimizer.zero_grad()
        output = aspect_model(**encoded)
        loss = aspect_loss_function(output.logits, labels_tensor)
        loss.backward()
        aspect_optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(batches)
    return average_loss


In [ ]:
def evaluate_aspects(texts, labels, threshold):
    aspect_model.eval()
    batches = make_batches(texts, labels, EVAL_BATCH_SIZE, shuffle=False)

    all_predictions = []
    all_true_labels = []

    with torch.no_grad():
        for batch_texts, batch_labels in batches:
            encoded = aspect_tokenizer(
                batch_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
            )
            encoded = {key: value.to(device) for key, value in encoded.items()}
            output = aspect_model(**encoded)
            probabilities = torch.sigmoid(output.logits).cpu().tolist()

            for probability_row in probabilities:
                predicted_row = []
                for probability in probability_row:
                    if probability >= threshold:
                        predicted_row.append(1)
                    else:
                        predicted_row.append(0)
                all_predictions.append(predicted_row)

            all_true_labels.extend(batch_labels)

    macro_f1 = f1_score(all_true_labels, all_predictions, average="macro", zero_division=0)
    micro_f1 = f1_score(all_true_labels, all_predictions, average="micro", zero_division=0)
    return macro_f1, micro_f1


### Train the aspect model

In [ ]:
best_aspect_macro_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_aspect_one_epoch(train_texts, train_aspect_labels)
    val_macro_f1, val_micro_f1 = evaluate_aspects(validation_texts, validation_aspect_labels, threshold=0.5)

    print(f"Epoch {epoch}: train loss = {train_loss:.4f}, "
          f"validation macro-F1 = {val_macro_f1:.4f}, validation micro-F1 = {val_micro_f1:.4f}")

    if val_macro_f1 > best_aspect_macro_f1:
        best_aspect_macro_f1 = val_macro_f1
        aspect_model.save_pretrained(ASPECT_DIR)
        aspect_tokenizer.save_pretrained(ASPECT_DIR)
        print("  New best aspect model saved.")


In [ ]:
aspect_model = AutoModelForSequenceClassification.from_pretrained(ASPECT_DIR)
aspect_model.to(device)

test_macro_f1, test_micro_f1 = evaluate_aspects(test_texts, test_aspect_labels, threshold=0.5)
print(f"Aspect test macro-F1 = {test_macro_f1:.4f}, test micro-F1 = {test_micro_f1:.4f}")


## Step 7: Pick a better aspect threshold

So far we called a probability "yes" whenever it was at least 0.5. Here we
try a small range of thresholds on the validation set and keep whichever one
gives the best macro-F1.


In [ ]:
best_threshold = 0.5
best_threshold_macro_f1 = -1.0

threshold = 0.20
while threshold <= 0.80:
    macro_f1, micro_f1 = evaluate_aspects(validation_texts, validation_aspect_labels, threshold=round(threshold, 2))
    print(f"threshold = {threshold:.2f}, validation macro-F1 = {macro_f1:.4f}")

    if macro_f1 > best_threshold_macro_f1:
        best_threshold_macro_f1 = macro_f1
        best_threshold = round(threshold, 2)

    threshold += 0.05

print("Best threshold:", best_threshold)


In [ ]:
threshold_info = {"global_threshold": best_threshold}
with open(ASPECT_DIR / "threshold.json", "w") as f:
    json.dump(threshold_info, f, indent=2)


## Step 8: Try the models out

Both models are still in memory from training above, so we can use them
right away. `kaggle_infer.ipynb` does the same thing but loads the models
fresh from saved files, for a later session.


In [ ]:
def predict_sentiment(text):
    sentiment_model.eval()
    encoded = sentiment_tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        output = sentiment_model(**encoded)

    probabilities = torch.softmax(output.logits, dim=1)[0].cpu().tolist()
    scores = {}
    for i in range(len(SENTIMENT_LABELS)):
        scores[SENTIMENT_LABELS[i]] = round(probabilities[i], 4)

    predicted_id = int(torch.argmax(output.logits, dim=1)[0])
    predicted_label = SENTIMENT_LABELS[predicted_id]

    return predicted_label, scores


In [ ]:
def predict_aspects(text):
    aspect_model.eval()
    encoded = aspect_tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        output = aspect_model(**encoded)

    probabilities = torch.sigmoid(output.logits)[0].cpu().tolist()

    detected_aspects = []
    for i in range(len(ASPECT_COLUMNS)):
        if probabilities[i] >= best_threshold:
            detected_aspects.append((ASPECT_COLUMNS[i], round(probabilities[i], 4)))

    detected_aspects.sort(key=lambda item: item[1], reverse=True)
    return detected_aspects


In [ ]:
example_sentences = [
    "The staff were friendly and the room was spotless.",
    "The Wi-Fi was slow and the room was noisy.",
    "Breakfast was poor but the staff were excellent.",
    "The hotel is located three kilometres from the airport.",
]

for sentence in example_sentences:
    predicted_label, scores = predict_sentiment(sentence)
    aspects_found = predict_aspects(sentence)

    print("Review:", sentence)
    print("Sentiment:", predicted_label, scores)
    print("Aspects:", aspects_found)
    print()
